In [1]:

%matplotlib inline
import jax.numpy as jnp
from jax import value_and_grad
from jax import grad
from jax import random
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as snb
import numpy as np
from scipy.stats import norm as norm_dist

###Bayesian ML functions###

from bayesian_ml import *
from packages.BayesianLinearRegression import BayesianLinearRegression
from packages.LogisticRegression import LogisticRegression
from packages.Grid2D import Grid2D
from packages.LaplaceApproximation import LaplaceApproximation
from packages.PosteriorPredictiveDistribution import PosteriorPredictiveDistribution
from packages.Hyperparameters import Hyperparameters
from packages.StationaryIsotropicKernel import StationaryIsotropicKernel
from packages.GaussianProcessRegression import GaussianProcessRegression
from packages.BayesianLinearSoftmax import BayesianLinearSoftmax
from packages.metropolis import metropolis

from IPython.display import Markdown, display


###Distributions###
from scipy.stats import multivariate_normal as mvn
from scipy.stats import poisson

from packages.util_funs import sigmoid
from packages.util_funs import log_npdf

snb.set_theme(font_scale=1.25)
key = random.key(seed = 0)

In [12]:
def to_latex_matrix(matrix):
    """Convert a JAX/numpy matrix to a LaTeX bmatrix string."""
    rows = []
    for row in matrix:
        rows.append(" & ".join(f"{val:.2f}" for val in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"

def to_latex_vector(vector):
    """Convert a JAX/numpy vector to a LaTeX bmatrix column vector."""
    rows = r" \\ ".join(f"{val:.2f}" for val in vector)
    return r"\begin{bmatrix}" + rows + r"\end{bmatrix}"

probit = lambda x: norm_dist.cdf(x)

# Part 1

## Question 1.1

# Part 2

## Question 2.1

The **squared exponential** covariance function is given by

\begin{align*} 
    k_{\text{SE}}(\mathbf{x}_n, \mathbf{x}_m) = \kappa^2 \exp\left(-\frac{\|\mathbf{x}_n - \mathbf{x}_m\|^2_ 2}{2\ell^2}\right), \tag{3}
\end{align*}

where $\kappa > 0$ and $\ell > 0$ are hyperparameters of the kernel.

Thus from eq. (7) the parameters can be read as $\kappa = \sqrt{2}$ and $\ell = 2$

## Question 2.2

In [15]:
from bayesian_ml import generate_samples
import jax.numpy as jnp
class GaussianProcessRegression(object):

    def __init__(self, X, y, kernel, hyperparameters, jitter=1e-8):
        """  
        Arguments:
            X                -- NxD input points
            y                -- Nx1 observed values 
            kernel           -- must be instance of the StationaryIsotropicKernel class
            jitter           -- non-negative scaler
            hyperparameters  -- Hyperparameter object containing kernel hyperparameters and noise std. dev. 
        """
        self.X = X
        self.y = y
        self.N = len(X)
        self.kernel = kernel
        self.jitter = jitter
        self.set_hyperparameters(hyperparameters)
        self.check_dimensions()

    def check_dimensions(self):
        N, D = self.X.shape
        assert self.X.ndim == 2, f"The variable X must be of shape (N, D), however, the current shape is: {self.X.shape}"
        assert self.y.ndim == 2, f"The varabiel y must be of shape (N, 1), however. the current shape is: {self.y.shape}"
        assert self.y.shape == (N, 1), f"The varabiel y must be of shape (N, 1), however. the current shape is: {self.y.shape}"

    def set_hyperparameters(self, hyper):
        self.hyperparameters = hyper
        
    def posterior_samples(self, key, Xstar, num_samples):
        """
            generate samples from the posterior p(f^*|y, x^*) for each of the inputs in Xstar

            Arguments:
                key              -- jax random key for controlling the random number generator
                Xstar            -- PxD prediction points
        
            returns:
                f_samples        -- numpy array of (P, num_samples) containing num_samples for each of the P inputs in Xstar
        """
        ##############################################
        # Your solution goes here
        ##############################################
        
        mu, Sigma = self.predict_f(Xstar)
        f_samples = generate_samples(key, mu.ravel(), Sigma, num_samples)
        
        ##############################################
        # End of solution
        ##############################################

        assert (f_samples.shape == (len(Xstar), num_samples)), f"The shape of the posterior mu seems wrong. Expected ({len(Xstar)}, {num_samples}), but actual shape was {f_samples.shape}. Please check implementation"
        return f_samples
        
    def predict_y(self, Xstar):
        """ returns the posterior distribution of y^* evaluated at each of the points in x^* conditioned on (X, y)
        
        Arguments:
        Xstar            -- PxD prediction points
        
        returns:
        mu               -- Px1 mean vector
        Sigma            -- PxP covariance matrix
        """

        ##############################################
        # Your solution goes here
        ##############################################
        
        # prepare relevant matrices
        mu, Sigma = self.predict_f(Xstar)
        Sigma = Sigma + self.hyperparameters.sigma**2 * jnp.identity(len(mu))
        
        ##############################################
        # End of solution
        ##############################################

        return mu, Sigma

    def predict_f(self, Xstar):
        """ returns the posterior distribution of f^* evaluated at each of the points in x^* conditioned on (X, y)
        
        Arguments:
        Xstar            -- PxD prediction points
        
        returns:
        mu               -- Px1 mean vector
        Sigma            -- PxP covariance matrix
        """

        ##############################################
        # Your solution goes here
        ##############################################
        
        # prepare relevant matrices
        k = self.kernel.construct_kernel(Xstar, self.X, self.hyperparameters, jitter=self.jitter)
        K = self.kernel.construct_kernel(self.X, self.X, self.hyperparameters, jitter=self.jitter)
        Kstar = self.kernel.construct_kernel(Xstar, Xstar, self.hyperparameters, jitter=self.jitter)
        
        # Compute C matrix
        C = K + self.hyperparameters.sigma**2*jnp.identity(len(self.X)) 

        # computer mean and Sigma
        mu = jnp.dot(k, jnp.linalg.solve(C, self.y))
        Sigma = Kstar - jnp.dot(k, jnp.linalg.solve(C, k.T))
        
        ##############################################
        # End of solution
        ##############################################

        # sanity check for dimensions
        assert (mu.shape == (len(Xstar), 1)), f"The shape of the posterior mu seems wrong. Expected ({len(Xstar)}, 1), but actual shape was {mu.shape}. Please check implementation"
        assert (Sigma.shape == (len(Xstar), len(Xstar))), f"The shape of the posterior Sigma seems wrong. Expected ({len(Xstar)}, {len(Xstar)}), but actual shape was {Sigma.shape}. Please check implementation"

        return mu, Sigma
    
    def log_marginal_likelihood(self, hyperparameters):
        """ 
            evaluate the log marginal likelihood p(y) given the hyperparaemters 

            Arguments:
                hyperparameters  -- Hyperparameter object containing kernel hyperparameters and noise std. dev. 
            """

        ##############################################
        # Your solution goes here
        ##############################################
        
        # prepare kernels
        K = self.kernel.construct_kernel(self.X, self.X, hyperparameters)
        C = K + hyperparameters.sigma**2*jnp.identity(self.N)

        # compute Cholesky decomposition
        L = jnp.linalg.cholesky(C)
        v = jnp.linalg.solve(L, self.y)

        # compute log marginal likelihood
        logdet_term = jnp.sum(jnp.log(jnp.diag(L)))
        quad_term =  0.5*jnp.sum(v**2)
        const_term = -0.5*self.N*jnp.log(2*jnp.pi)

        return const_term - logdet_term - quad_term
        
        ##############################################
        # End of solution
        ##############################################

Xtrain = jnp.array([-2, 0, 2])[:,None]
ytrain = jnp.array([-2.01, 1.41, 0.23])[:,None]

def squared_exponential(tau, hyperparameters):
    return hyperparameters.kappa**2*jnp.exp(-0.5*tau**2/hyperparameters.lengthscale**2)

# specify hyperparameters
hyper = Hyperparameters(kappa=jnp.sqrt(2), lengthscale=2, sigma=1/2)

# instantiate  kernel objects
kernel = StationaryIsotropicKernel(squared_exponential)

# instantiate GP without data (hence, posterior=prior) and with data
gp = GaussianProcessRegression(Xtrain, ytrain, kernel, hyper)

K = gp.kernel.construct_kernel(Xtrain, Xtrain, hyper)
K

display(Markdown(rf"The prior distribution is $p(f \mid x) = \mathcal{{N}}(0, \bf{{K}})= \mathcal{{N}}(0, {to_latex_matrix(K)})$"))

The prior distribution is $p(f \mid x) = \mathcal{N}(0, \bf{K})= \mathcal{N}(0, \begin{bmatrix}2.00 & 1.21 & 0.27 \\ 1.21 & 2.00 & 1.21 \\ 0.27 & 1.21 & 2.00\end{bmatrix})$

## Question 2.3

The posterior distribution is given by

$$p(f | \bf{y}) = \mathcal{N}(f | \mu_{f | \bf{y}}, \sigma^{2}_{f | \bf{y}})$$
where

$$\mu_{f^* \mid y} = \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{y}$$
and
$$\sigma^2_{f \mid y} = \bf{K} - \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{k}^T$$


In [20]:
m,S = gp.predict_f(Xtrain)

display(Markdown(rf"The posterior distribution is $p(f \mid y, x) = \mathcal{{N}}({to_latex_vector(m.flatten())}, {to_latex_matrix(S)})$"))

The posterior distribution is $p(f \mid y, x) = \mathcal{N}(\begin{bmatrix}-1.53 \\ 0.89 \\ 0.43\end{bmatrix}, \begin{bmatrix}0.21 & 0.03 & -0.01 \\ 0.03 & 0.19 & 0.03 \\ -0.01 & 0.03 & 0.21\end{bmatrix})$

## Question 2.4

\begin{aligned}
&\operatorname{Var}[f(x)]=k_2(x, x)=\exp \left(-\frac{1}{2}\|0-0\|_2\right)+2=1+2=3\\
&\text { for all } x \in \mathbb{R} \text {. }
\end{aligned}

## Part 3:

## Question 3.1

In [27]:
class BayesianLinearRegression(object):
    
    def __init__(self, Phi, y, alpha=1., beta=1.):
        
        # store data and hyperparameters
        self.Phi, self.y = Phi, y
        self.N, self.D = Phi.shape
        self.alpha, self.beta = alpha, beta
        
        # compute posterior distribution
        self.m, self.S = self.compute_posterior(alpha, beta)
        self.log_marginal_likelihood = self.compute_marginal_likelihood(alpha, beta)

        # perform sanity check of shapes/dimensions
        self.check_dimensions()
        
    def w_MLE(self):
        return jnp.linalg.solve(self.Phi.T@self.Phi, self.Phi.T@self.y).ravel()

    def w_MAP(self, dim):
        return (self.beta*jnp.linalg.solve(self.alpha*jnp.identity(dim) + self.beta*(self.Phi.T@self.Phi), self.Phi.T)@self.y).ravel()
    
    def check_dimensions(self):
        D = self.D
        assert self.y.shape == (self.N, 1), f"Wrong shape for data vector y.\n For N = {self.N}, the shape of y must be ({self.N}, 1), but the actual shape is {self.y.shape}"
        assert self.m.shape == (D, 1), f"Wrong shape for posterior mean.\nFor D = {D}, the shape of the posterior mean must be ({D}, 1), but the actual shape is {self.m.shape}"
        assert self.S.shape == (D, D), f"Wrong shape for posterior covariance.\nFor D = {D}, the shape of the posterior mean must be ({D}, {D}), , but the actual shape is {self.S.shape}"

    def compute_posterior(self, alpha, beta):
        """ computes the posterior N(w|m, S) and return m, S.
            Shape of m and S must be (D, 1) and (D, D), respectively  """
        
        #############################################
        # Insert your solution here
        #############################################
        
        # compute prior and posterior precision 
        inv_S0 = alpha*jnp.identity(self.D)
        A = inv_S0 + beta*(self.Phi.T@self.Phi)
        
        # compute mean and covariance 
        m = beta*jnp.linalg.solve(A, self.Phi.T)@self.y   # eq. (2) above
        S = jnp.linalg.inv(A)                             # eq. (1) above
        
        #############################################
        # End of solution
        #############################################
        return m, S
      
    def generate_prior_samples(self, key, num_samples):
        """ generate samples from the prior  """
        return random.multivariate_normal(key, jnp.zeros(len(self.m)), (1/self.alpha)*jnp.identity(len(self.m)), shape=(num_samples, ))
    
    def generate_posterior_samples(self, key, num_samples):
        """ generate samples from the posterior  """
        return random.multivariate_normal(key, self.m.ravel(), self.S, shape=(num_samples, ))
    
    def predict_f(self, Phi):
        """ computes posterior mean (mu_f) and variance (var_f) of f(phi(x)) for each row in Phi-matrix.
            If Phi is a [N, D]-matrix, then the shapes of both mu_f and var_f must be (N,)
            The function returns (mu_f, var_f)
        """
        mu_f = (Phi@self.m).ravel()   
        var_f = jnp.diag(Phi@self.S@Phi.T)   
        
        # check dimensions before returning values
        assert mu_f.shape == (Phi.shape[0],), "Shape of mu_f seems wrong. Check your implementation"
        assert var_f.shape == (Phi.shape[0],), "Shape of var_f seems wrong. Check your implementation"
        return mu_f, var_f
    
    
        
    def predict_y(self, Phi):
        """ returns posterior predictive mean (mu_y) and variance (var_y) of y = f(phi(x)) + e for each row in Phi-matrix.
            If Phi is a [N, D]-matrix, then the shapes of both mu_y and var_y must be (N,).
            The function returns (mu_y, var_y)
        """
        mu_f, var_f = self.predict_f(Phi)
        mu_y = mu_f                  
        var_y = var_f + 1/self.beta  

        # check dimensions before returning values
        assert mu_y.shape == (Phi.shape[0],), "Shape of mu_y seems wrong. Check your implementation"
        assert var_y.shape == (Phi.shape[0],), "Shape of var_y seems wrong. Check your implementation"
        return mu_y, var_y
        
    
    def compute_marginal_likelihood(self, alpha, beta):
        """ computes and returns log marginal likelihood p(y|alpha, beta) """
        inv_S0 = alpha*jnp.identity(self.D)
        A = inv_S0 + beta*(self.Phi.T@self.Phi)
        m = beta*jnp.linalg.solve(A, self.Phi.T)@self.y   # (eq. 3.53 in Bishop)
        S = jnp.linalg.inv(A)                             # (eq. 3.54 in Bishop)
        Em = beta/2*jnp.sum((self.y - self.Phi@m)**2) + alpha/2*jnp.sum(m**2)
        return self.D/2*jnp.log(alpha) + self.N/2*jnp.log(beta) - Em - 0.5*jnp.linalg.slogdet(A)[1] - self.N/2*jnp.log(2*jnp.pi)
         

    def optimize_hyperparameters(self):
        # optimizes hyperparameters using marginal likelihood
        theta0 = jnp.array((jnp.log(self.alpha), jnp.log(self.beta)))
        def negative_marginal_likelihood(theta):
            alpha, beta = jnp.exp(theta[0]), jnp.exp(theta[1])
            return -self.compute_marginal_likelihood(alpha, beta)

        result = minimize(value_and_grad(negative_marginal_likelihood), theta0, jac=True)

        # store new hyperparameters and recompute posterior
        theta_opt = result.x
        self.alpha, self.beta = jnp.exp(theta_opt[0]), jnp.exp(theta_opt[1])
        self.m, self.S = self.compute_posterior(self.alpha, self.beta)
        self.log_marginal_likelihood = self.compute_marginal_likelihood(self.alpha, self.beta)

    def predict_f_MLE(self, Phi):
        """Point-estimate approximation of f using w_MLE.
        Since there's no weight distribution, var_f is identically zero."""
        w = self.w_MLE()                          # shape (D,)
        mu_f = (Phi @ w).ravel()                  # shape (N,)
        var_f = jnp.zeros(Phi.shape[0])           # no epistemic uncertainty
        return mu_f, var_f

    def predict_y_MLE(self, Phi):
        """Point-estimate predictive distribution using w_MLE.
        var_y captures only noise (1/beta), not weight uncertainty."""
        mu_f, _ = self.predict_f_MLE(Phi)
        mu_y = mu_f
        var_y = jnp.full(Phi.shape[0], 1/self.beta)
        return mu_y, var_y

    def predict_f_MAP(self, Phi):
        """Point-estimate approximation of f using w_MAP."""
        w = self.w_MAP(self.D)                    # uses stored alpha, beta, D
        mu_f = (Phi @ w).ravel()
        var_f = jnp.zeros(Phi.shape[0])
        return mu_f, var_f

    def predict_y_MAP(self, Phi):
        """Point-estimate predictive distribution using w_MAP."""
        mu_f, _ = self.predict_f_MAP(Phi)
        mu_y = mu_f
        var_y = jnp.full(Phi.shape[0], 1/self.beta)
        return mu_y, var_y

m = jnp.array([1, 1])
tau2 = 1
sigma2 = 1
Xtrain = jnp.array([[1, 0.5], [-1, 1]])
ytrain = jnp.array([1, 0])[:,None]
beta = 1 / sigma2
alpha = 1 / tau2

BR = BayesianLinearRegression(Xtrain, ytrain, alpha, beta)
BR.w_MLE()

Array([0.66666667, 0.66666667], dtype=float64)

## Question 3.2

In [22]:
theta_prior = jnp.array([0, 0])

1/2*mvn.pdf(theta_prior, -m, tau2 * jnp.identity(len(m))) + 1/2*mvn.pdf(theta_prior, m, tau2 * jnp.identity(len(m)))

np.float64(0.05854983152431917)